# **Project Name**    -



##### **Project Type**    - EDA + Regression
##### **Contribution**    - Individual


# **Project Summary -**

The Glassdoor Job Salary Prediction project is a comprehensive machine learning application aimed at forecasting average salaries for various job listings using data scraped from Glassdoor. The dataset includes job titles, salary estimates, company attributes (such as size, revenue, and industry), and geographical information. The dataset was preprocessed by cleaning and transforming features such as salary estimates, company size, revenue, and company age, handling missing values, and removing irrelevant columns like job descriptions and competitors. Key steps included converting salary ranges to average values, encoding categorical variables, and calculating company age from founding years.

Hyperparameter tuning and cross-validation are applied to improve model performance. The final model is selected based on its ability to generalize well and provide accurate salary predictions. In summary, this project demonstrates a full machine learning pipeline—from data preprocessing and EDA to model training and deployment—and serves as a strong portfolio piece for showcasing practical data science and regression modeling skills.

# **GitHub Link -**

Provide your GitHub Link here.

# **Problem Statement**


The objective of this project is to predict the average salary for job listings based on various attributes provided in a Glassdoor job dataset. This project addresses the challenge of accurately predicting job salaries in a competitive labor market, where factors like job title, company size, age, rating, and ownership type influence compensation but are often inconsistently reported or missing in datasets. By leveraging a Glassdoor dataset, the project aims to build a regression model that identifies key salary predictors, handles data inconsistencies, and provides reliable salary estimates to support better decision-making in recruitment and career planning.

# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries

import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Model Selection
from sklearn.model_selection import train_test_split, GridSearchCV

# Models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

### Dataset Loading

In [ ]:
# Load Dataset
df = pd.read_csv('/content/glassdoor_jobs.csv')

### Dataset First View

In [ ]:
# Dataset First Look
df.head()

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
df.shape

### Dataset Information

In [ ]:
# Dataset Info
df.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
df.duplicated().sum()

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
df.isnull().sum()

In [ ]:
# Visualizing the missing values
plt.figure(figsize=(8,4))
sns.heatmap(df.isnull(), cbar=False, cmap="viridis")
plt.title("Missing Values Heatmap")
plt.show()

### What did you know about your dataset?

The dataset contains job-related information such as job title, salary estimates, company details, location, and industry. It includes both categorical and numerical features. On preliminary examination, we can see that in the dataset there are no missing or duplicate values.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
df.columns

In [ ]:
# Dataset Describe
df.describe()

### Variables Description

There are 15 feature in the given dataset. .describe() method gives the statistical analysis of numerical columns we have in the given dataset. .nunique() method checks the number of unique values in each column/variable.

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
df.nunique()

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Create Seniority Feature
def seniority(title):
    if 'senior' in title.lower():
        return 'Senior'
    elif 'junior' in title.lower() or 'jr' in title.lower():
        return 'Junior'
    else:
        return 'Mid'

df['seniority'] = df['Job Title'].apply(seniority)

# Extract State
df['job_state'] = df['Location'].apply(lambda x: x.split(',')[-1])

# Company Age
df['company_age'] = df['Founded'].apply(lambda x: 2026 - x if x > 0 else None)



# Cleaning salary column
df = df[df['Salary Estimate'] != '-1']  # remove rows where salary is -1
df = df[~df['Salary Estimate'].str.contains('per hour', case=False)]
df = df[~df['Salary Estimate'].str.contains('employer provided', case=False)]

df['Salary Estimate'] = df['Salary Estimate'].str.replace('$', '')
df['Salary Estimate'] = df['Salary Estimate'].str.replace('K', '')
df['Salary Estimate'] = df['Salary Estimate'].str.replace('Employer Provided Salary:', '')
df['Salary Estimate'] = df['Salary Estimate'].str.split('(').str[0]

# split into min and max
df[['min_salary', 'max_salary']] = df['Salary Estimate'].str.split('-', expand=True)

# convert to numbers
df['min_salary'] = df['min_salary'].astype(int)
df['max_salary'] = df['max_salary'].astype(int)

# Create average salary
df['avg_salary'] = (df['min_salary'] + df['max_salary']) / 2


# Remove unnecessary columns
del df['Unnamed: 0']
del df['Competitors']
del df['Salary Estimate']


In [ ]:
df.info()

### What all manipulations have you done and insights you found?

Several data preprocessing and feature engineering steps were performed to make the dataset suitable for analysis and modeling.

- The salary column was cleaned by removing text and extracting minimum and maximum salary values. A new feature called average salary was created for better prediction.
- A new feature called seniority was created from job titles to categorize roles into Junior, Mid, and Senior levels.
- Location data was processed to extract state information for regional analysis.
- Company age was derived from the founding year to understand the impact of company maturity on salaries.
- Irrelevant columns were removed to reduce noise in the dataset.


## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart - 1 visualization code
plt.figure(figsize=(8,5))
sns.histplot(df['avg_salary'], bins=30, kde=True)
plt.title("Distribution of Average Salary")
plt.xlabel("Average Salary")
plt.ylabel("Frequency")
plt.show()

##### 1. Why did you pick the specific chart?


Histogram helps to understand the distribution of salary and identify whether it is skewed or normally distributed.


##### 2. What is/are the insight(s) found from the chart?

Most salaries are concentrated in a mid-range, with fewer jobs offering extremely high salaries. The distribution appears slightly right-skewed

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.


This insight helps job seekers understand the common salary range and set realistic expectations.

#### Chart - 2

In [ ]:
# Chart - 2 visualization code
top_jobs = df['Job Title'].value_counts().head(10).index

plt.figure(figsize=(10,6))
sns.boxplot(x='avg_salary', y='Job Title', data=df[df['Job Title'].isin(top_jobs)])
plt.title("Salary by Job Title")
plt.show()

##### 1. Why did you pick the specific chart?

Boxplot shows salary variation across job roles.

##### 2. What is/are the insight(s) found from the chart?


Different roles have significantly different salary ranges. Senior roles tend to have higher median salaries.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.


Helps job seekers choose high-paying roles and helps companies benchmark salaries.

#### Chart - 3

In [ ]:
# Chart - 3 visualization code
plt.hist(df['avg_salary'], bins=20, alpha=0.7, color='skyblue', edgecolor='black')
plt.axvline(df['avg_salary'].mean(), color='red', linestyle='--',
            label=f'Mean: ${df["avg_salary"].mean():.1f}k')
plt.axvline(df['avg_salary'].median(), color='green', linestyle='--',
            label=f'Median: ${df["avg_salary"].median():.1f}k')
plt.xlabel('Salary Estimate (k$)')
plt.ylabel('Frequency')
plt.title(' Salary Distribution', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

##### 1. Why did you pick the specific chart?

A histogram was selected to show the distribution of Salary Estimates, highlighting the spread and central tendencies for salary prediction analysis.

##### 2. What is/are the insight(s) found from the chart?

The Salary Estimate distribution is right-skewed, with a mean of  102.4k and medianof 98.0k, indicating most salaries are below the mean due to a few high outliers.



##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Understanding the skewed salary distribution can help businesses set competitive salary ranges and assist job seekers in negotiating salaries, potentially improving hiring strategies and satisfaction.

#### Chart - 4

In [ ]:
# Chart - 4 visualization code

plt.figure(figsize=(8,5))
sns.scatterplot(x='Rating', y='avg_salary', data=df)
plt.title("Salary vs Company Rating")
plt.show()

##### 1. Why did you pick the specific chart?


Scatter plot helps visualize the relationship between company ratings and salary.

##### 2. What is/are the insight(s) found from the chart?

There is no strong direct correlation between company rating and salary, though higher-rated companies sometimes offer slightly better pay.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Employees should not rely solely on ratings when evaluating salary expectations. Companies may improve compensation to match their reputation.

#### Chart - 5

In [ ]:
# Chart - 5 visualization code
top_ind = df['Industry'].value_counts().head(10).index

plt.figure(figsize=(10,6))
sns.boxplot(x='Industry', y='avg_salary', data=df[df['Industry'].isin(top_ind)])
plt.xticks(rotation=45 , ha='right')
plt.title("Salary by Industry")

plt.show()

##### 1. Why did you pick the specific chart?


To compare salary differences across industries.

##### 2. What is/are the insight(s) found from the chart?


Certain industries (like tech or finance) offer higher salaries compared to others

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.


Professionals can choose industries strategically, and companies must stay competitive within their sector.

#### Chart - 6

In [ ]:
# Chart - 6 visualization code

sns.boxplot(x='seniority', y='avg_salary', data=df)
plt.title("Salary by Seniority")
plt.show()

##### 1. Why did you pick the specific chart?

Boxplot shows salary differences across seniority levels.

##### 2. What is/are the insight(s) found from the chart?

Senior roles clearly earn higher salaries compared to junior and mid-level roles.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This highlights career growth potential and encourages skill development for higher pay.



#### Chart - 07

In [ ]:
# Chart - 7 visualization code

plt.figure(figsize=(10,6))
sns.boxplot(x='Size', y='avg_salary', data=df)
plt.xticks(rotation=45)
plt.title("Salary vs Company Size")
plt.show()

##### 1. Why did you pick the specific chart?


Boxplot helps in comparing salary distributions across different company sizes

##### 2. What is/are the insight(s) found from the chart?

Larger companies tend to offer higher median salaries compared to smaller companies. However, there is variability within each category.



##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.


This suggests that job seekers targeting higher salaries may prefer larger organizations. Companies can benchmark their compensation strategies accordingly.

#### Chart - 8 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

##### 1. Why did you pick the specific chart?


Heatmap shows correlation between numerical variables.

##### 2. What is/are the insight(s) found from the chart?


Average salary shows correlation with some features like company age and rating, but not strongly.

#### Chart - 9 - Pair Plot

In [ ]:
# Pair Plot visualization code
sns.pairplot(df[['avg_salary', 'Rating', 'company_age']])
plt.show()

##### 1. Why did you pick the specific chart?


Pairplot helps visualize relationships between multiple variables.

##### 2. What is/are the insight(s) found from the chart?


Some weak relationships exist between salary and other numerical features, indicating need for advanced models.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define hypothetical statements from the dataset and perform the hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

Based on the exploratory data analysis, the following hypotheses are formulated:

1. Salary differs significantly across job seniority levels.
2. Company size has a significant impact on salary.

### Hypothetical Statement - 1: Salary vs Seniority

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Null Hypothesis (H0): There is no significant difference in salary across different seniority levels.

Alternate Hypothesis (H1): There is a significant difference in salary across different seniority levels.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
from scipy.stats import f_oneway

junior = df[df['seniority'] == 'Junior']['avg_salary']
mid = df[df['seniority'] == 'Mid']['avg_salary']
senior = df[df['seniority'] == 'Senior']['avg_salary']

f_stat, p_value = f_oneway(junior, mid, senior)

print("P-value:", p_value)

##### Which statistical test have you done to obtain P-Value?

ANOVA (Analysis of Variance) test.

##### Why did you choose the specific statistical test?


ANOVA is used because we are comparing the mean salary across more than two groups (Junior, Mid, Senior).

### Hypothetical Statement - 2: Salary vs Company Size

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.


Null Hypothesis (H0): Company size has no significant impact on salary.

Alternate Hypothesis (H1): Company size significantly affects salary.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value

sizes = df['Size'].unique()
groups = [df[df['Size'] == size]['avg_salary'] for size in sizes]

from scipy.stats import f_oneway

f_stat, p_value = f_oneway(*groups)

print("P-value:", p_value)

##### Which statistical test have you done to obtain P-Value?

ANOVA Test


##### Why did you choose the specific statistical test?


Because salary is being compared across multiple categories of company size.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation
# print(df.isnull().sum()) # Null values in Column "Company Age"
df['company_age'] = df['company_age'].fillna(df['company_age'].median())  # Median Imputation
df.isnull().sum()

#### What all missing value imputation techniques have you used and why did you use those techniques?

Missing values already treated earlier during data wrangling steps

### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments

# Using IQR method
Q1 = df['avg_salary'].quantile(0.25)
Q3 = df['avg_salary'].quantile(0.75)
IQR = Q3 - Q1

df = df[(df['avg_salary'] >= Q1 - 1.5*IQR) & (df['avg_salary'] <= Q3 + 1.5*IQR)]

##### What all outlier treatment techniques have you used and why did you use those techniques?


The IQR (Interquartile Range) method has been used to detect and remove outliers. This method is effective for skewed data and ensures that extreme salary values do not distort model performance.

### 3. Categorical Encoding

In [ ]:
# Encode your categorical columns

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col].astype(str))

#### What all categorical encoding techniques have you used & why did you use those techniques?


Label Encoding was used to convert categorical variables into numerical format

### 4. Feature Manipulation and Selection


#### 1. Feature Manipulation

In [ ]:
# Extracting Skills from Job Description
df['Job Description'] = df['Job Description'].astype(str)
df['python'] = df['Job Description'].str.lower().apply(lambda x: 1 if 'python' in x else 0)
df['sql'] = df['Job Description'].str.lower().apply(lambda x: 1 if 'sql' in x else 0)
df['excel'] = df['Job Description'].str.lower().apply(lambda x: 1 if 'excel' in x else 0)
df['machine_learning'] = df['Job Description'].str.lower().apply(lambda x: 1 if 'machine learning' in x else 0)



In [ ]:
df.info()


#### 2. Feature Selection

In [ ]:
# Select your features wisely to avoid overfitting
X = df.drop('avg_salary', axis=1)
y = df['avg_salary']

##### What all feature selection methods have you used  and why?

Feature selection was done by removing irrelevant and redundant columns. Only meaningful features that contribute to salary prediction were retained.

##### Which all features you found important and why?

Features like job title, company size, industry, location, and seniority were found important as they directly influence salary levels.

### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

No major transformation was required as the dataset was already cleaned and structured. Feature engineering was sufficient for model readiness

### 6. Data Scaling

In [ ]:
# Scaling your data

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

##### Which method have you used to scale you data and why?


StandardScaler was used to normalize the feature values. It ensures that all features contribute equally to the model performance.

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?


Dimensionality reduction is not required as the number of features is manageable and does not cause high computational complexity

### 8. Data Splitting

In [ ]:
# Split your data to train and test. Choose Splitting ratio wisely.
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

##### What data splitting ratio have you used and why?


An 80-20 split was used, where 80% data is used for training and 20% for testing. This ensures sufficient data for training while maintaining a reliable test set.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.


Imbalanced dataset handling is not applicable as this is a regression problem, not a classification problem.

## ***7. ML Model Implementation***

### ML Model - 1 : Linear Regression

In [ ]:
# ML Model - 1 Implementation
lr = LinearRegression()

# Fit the model
lr.fit(X_train, y_train)

# Predict
y_pred_lr = lr.predict(X_test)

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
# Evaluation Metrics

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print("MAE:", mae_lr)
print("RMSE:", rmse_lr)
print("R2 Score:", r2_lr)

In [ ]:
# Visualizing evaluation Metric Score chart

metrics = ['MAE', 'RMSE', 'R2']
values = [mae_lr, rmse_lr, r2_lr]

plt.bar(metrics, values)
plt.title("Linear Regression Performance")
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

# Fit the Algorithm

# Predict on the model

##### Which hyperparameter optimization technique have you used and why?

No major tuning for Linear Regression

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.


No major improvement observed

### ML Model - 2 : Gradient Boosting

In [ ]:
# ML Model - 2 Implementation
gb = GradientBoostingRegressor()

gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)



#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.


Gradient Boosting builds models sequentially to correct errors from previous models

In [ ]:
mae_gb = mean_absolute_error(y_test, y_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
r2_gb = r2_score(y_test, y_pred_gb)

print("MAE:", mae_gb)
print("RMSE:", rmse_gb)
print("R2 Score:", r2_gb)

In [ ]:
# Visualization
metrics = ['MAE', 'RMSE', 'R2']
values = [mae_gb, rmse_gb, r2_gb]

plt.bar(metrics, values)
plt.title("Gradient Boosting Performance")
plt.show()

### ML Model - 3 : Random Forest

In [ ]:
# ML Model - 2 Implementation

rf = RandomForestRegressor(random_state=42)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.


Random Forest is an ensemble learning method that combines multiple decision trees to improve accuracy and reduce overfitting.

It performs significantly better than Linear Regression, capturing non-linear relationships in salary prediction

In [ ]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("MAE:", mae_rf)
print("RMSE:", rmse_rf)
print("R2 Score:", r2_rf)

In [ ]:
# Visualization
metrics = ['MAE', 'RMSE', 'R2']
values = [mae_rf, rmse_rf, r2_rf]

plt.bar(metrics, values)
plt.title("Random Forest Performance")
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 3 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20],
    'min_samples_split': [2, 5]
}
grid = GridSearchCV(RandomForestRegressor(), param_grid, cv=3, scoring='r2')
grid.fit(X_train, y_train)
best_rf = grid.best_estimator_
y_pred_rf_tuned = best_rf.predict(X_test)

### Post Tuning

In [ ]:
r2_rf_tuned = r2_score(y_test, y_pred_rf_tuned)
print("Tuned R2:", r2_rf_tuned)

##### Which hyperparameter optimization technique have you used and why?


GridSearchCV was used to find the best combination of parameters

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.


Yes, model performance improved after tuning as R2 score increased.

### 1. Which Evaluation metrics did you consider for a positive business impact and why?


R2 Score, RMSE, and MAE were used. RMSE penalizes large errors, making it suitable for salary prediction. R2 Score indicates how well the model explains variance.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?


Random Forest Regressor was selected as the final model because it provided the highest accuracy and captured complex relationships in the data.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?


Feature importance from Random Forest shows that job title, company size, and location are the most important factors affecting salary prediction.

## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
# Save the File

import joblib

joblib.dump(best_rf, 'salary_prediction_model.pkl')

### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Load the File and predict unseen data
loaded_model = joblib.load('salary_prediction_model.pkl')

# Take one sample
sample = X_test[0].reshape(1, -1)
prediction = loaded_model.predict(sample)

print("Predicted Salary:", prediction)

# **Conclusion**


The project successfully developed a predictive model for job salaries, with Random Forest outperforming other models due to its ability to handle complex, non-linear relationships in the data. Key predictors included job title, company size, revenue, and location. Challenges such as missing data and inconsistent salary formats were addressed through imputation and transformation techniques. The findings highlight the importance of feature engineering in improving model accuracy and provide valuable insights for job seekers and employers in understanding salary determinants. Future improvements could involve incorporating additional features like job seniority or market trends to enhance prediction robustness.

###*~ Pulkit Narang*